In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
"""
============================================================
ETAPA 1 — Ingestão e Coleta de Dados
Pipeline: Priorização de Infraestrutura Urbana Preventiva

Fontes oficiais:
  - S2ID:     https://s2id.mi.gov.br/paginas/series/          (download manual)
  - CadÚnico: https://aplicacoes.mds.gov.br/sagi/portal/      (download direto)
  - IVS/IPEA: atlasivs_dadosbrutos_pt_v2.xlsx                 (Google Drive)
              Caminho: /content/drive/MyDrive/ENAP/atlasivs_dadosbrutos_pt_v2.xlsx

Requer: pip install requests pandas openpyxl
============================================================
"""

import zipfile, unicodedata, requests
import pandas as pd
from pathlib import Path
from datetime import datetime

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# ── Caminhos e URLs ───────────────────────────────────────────────────────────
IVS_XLSX_PATH = Path("/content/drive/MyDrive/ENAP/dados_pnpdec/atlasivs_dadosbrutos_pt_v2.xlsx")

COBRADE_CSV_PATH = Path("/content/drive/MyDrive/ENAP/dados_pnpdec/cobrade.csv")

RESPOSTA_XLSX_PATH = Path("/content/drive/MyDrive/ENAP/dados_pnpdec/relatorio_respostas.xlsx")


# CadÚnico — base amostral desidentificada dez/2018 (última versão pública disponível)
# Fonte: https://aplicacoes.mds.gov.br/sagicad/pesquisas/pes-metadados.php
# Para versões mais recentes, solicite acesso formal à SAGICAD/MDS
CADUNICO_URL = (
    "https://aplicacoes.mds.gov.br/sagi/pesquisas/redM.php"
    "?t=microdado&l=./documentos/microdado/base_amostra_cad_201812.zip"
)


# ── Utilitário ────────────────────────────────────────────────────────────────
def _norm(s: str) -> str:
    s = str(s).strip().lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return s.replace(" ", "_").replace("-", "_").replace("/", "_")


def download_arquivo(url, destino, descricao=""):
    destino = Path(destino)
    if destino.exists():
        print(f"[CACHE] {descricao} já existe.")
        return destino
    print(f"[DOWNLOAD] {descricao} ...")
    r = requests.get(url, timeout=300, stream=True)
    r.raise_for_status()
    with open(destino, "wb") as f:
        for chunk in r.iter_content(8192):
            f.write(chunk)
    print(f"[OK] {destino.stat().st_size / 1024:.0f} KB")
    return destino


# ── IVS ───────────────────────────────────────────────────────────────────────
def carregar_ivs(caminho_xlsx: Path) -> pd.DataFrame:
    """
    Lê o arquivo atlasivs_dadosbrutos_pt_v2.xlsx do IPEA.

    Estrutura real do arquivo:
      - 340k linhas com múltiplos níveis (brasil, uf, municipio, udh) e anos
      - Coluna 'nivel'       : filtra por 'municipio'
      - Coluna 'municipio'   : código IBGE com 6 dígitos (zfill para 7)
      - Coluna 'ano'         : usa o mais recente disponível (2010)
      - Colunas IVS          : ivs, ivs_infraestrutura_urbana,
                               ivs_capital_humano, ivs_renda_e_trabalho
    """
    caminho_xlsx = Path(caminho_xlsx)
    if not caminho_xlsx.exists():
        raise FileNotFoundError(
            f"Arquivo IVS não encontrado: {caminho_xlsx}\n"
            f"Certifique-se de que o Google Drive está montado:\n"
            f"  from google.colab import drive\n"
            f"  drive.mount('/content/drive')"
        )

    print(f"[IVS] Lendo {caminho_xlsx.name} (pode demorar alguns segundos)...")
    df = pd.read_excel(caminho_xlsx, dtype=str, engine="openpyxl")
    df.columns = [_norm(c) for c in df.columns]
    print(f"[IVS] Shape bruto: {df.shape}")

    # ── Filtra apenas nível município ─────────────────────────────────────────
    # O campo 'nivel' contém combinações como "regiao,uf,rm,municipio"
    if "nivel" not in df.columns:
        raise ValueError(f"Coluna 'nivel' não encontrada. Colunas: {list(df.columns)}")

    df = df[
        df["nivel"].str.strip().str.lower() == "regiao,uf,rm,municipio"
    ].copy()
    print(f"[IVS] Após filtro nivel=municipio: {len(df)} linhas")

    # ── Filtra ano 2010 (Censo — mais completo para IVS municipal) ────────────
    if "ano" in df.columns:
        df["ano"] = pd.to_numeric(df["ano"], errors="coerce")
        df = df[df["ano"] == 2010].copy()
        print(f"[IVS] Ano selecionado: 2010 | {len(df)} linhas")

    # ── Filtra apenas totais por cor (Total Cor) ──────────────────────────────
    # label_cor tem: "Total Cor", "Branco", "Negro"
    if "label_cor" in df.columns:
        df = df[df["label_cor"].str.strip() == "Total Cor"].copy()
        print(f"[IVS] Após filtro Total Cor: {len(df)} linhas")

    # ── Código IBGE ───────────────────────────────────────────────────────────
    # A coluna 'municipio' contém o código de 6 dígitos
    if "municipio" not in df.columns:
        raise ValueError(f"Coluna 'municipio' não encontrada: {list(df.columns)}")

    df = df.rename(columns={"municipio": "cod_ibge"})
    df["cod_ibge"] = df["cod_ibge"].astype(str).str.strip().str.zfill(7)
    df = df[df["cod_ibge"].str.match(r"^\d{7}$")].copy()

    # ── Colunas IVS ───────────────────────────────────────────────────────────
    # Nomes exatos conforme o arquivo real:
    #   ivs, ivs_infraestrutura_urbana, ivs_capital_humano, ivs_renda_e_trabalho
    renomear = {
        "ivs":                      "ivs_geral",
        "ivs_renda_e_trabalho":     "ivs_renda_trabalho",
        # já corretos: ivs_infraestrutura_urbana, ivs_capital_humano
    }
    df = df.rename(columns=renomear)

    colunas_ivs = [
        "ivs_geral",
        "ivs_infraestrutura_urbana",
        "ivs_capital_humano",
        "ivs_renda_trabalho",
    ]
    for c in colunas_ivs:
        if c in df.columns:
            df[c] = pd.to_numeric(
                df[c].astype(str).str.replace(",", "."), errors="coerce"
            )

    # Remove duplicatas de cod_ibge (mantém primeira ocorrência)
    df = df.drop_duplicates(subset="cod_ibge", keep="first")

    colunas_ok = ["cod_ibge"] + [c for c in colunas_ivs if c in df.columns]
    print(f"[IVS] {len(df)} municípios únicos | colunas: {colunas_ok}")
    print(f"[IVS] Amostra:\n{df[colunas_ok].head(3).to_string()}\n")
    return df[colunas_ok]


# ── S2ID ──────────────────────────────────────────────────────────────────────
def carregar_s2id(caminho_csv: Path) -> pd.DataFrame:
    """
    Lê a série histórica do S2ID exportada manualmente do portal.

    Como exportar:
      1. Acesse https://s2id.mi.gov.br/paginas/series/
      2. Selecione o período e exporte como CSV
      3. Salve em: data/raw/s2id_serie_historica.csv
    """
    df = pd.read_csv(caminho_csv, sep=";", encoding="latin-1", dtype=str)
    df.columns = [_norm(c) for c in df.columns]
    print(f"[S2ID] Colunas: {list(df.columns)}")

    col_ibge = next(
        (c for c in df.columns
         if "ibge" in c or c in ("co_municipio", "cod_municipio", "cod_ibge")),
        None
    )
    if col_ibge is None:
        raise ValueError(
            f"Coluna de código IBGE não encontrada no S2ID.\n"
            f"Colunas disponíveis: {list(df.columns)}"
        )
    df = df.rename(columns={col_ibge: "cod_ibge"})
    df["cod_ibge"] = df["cod_ibge"].str.strip().str.zfill(7)

    for col in ["afetados", "mortos", "desalojados", "desabrigados"]:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col].str.replace(",", "."), errors="coerce"
            ).fillna(0)

    col_cobrade = next((c for c in df.columns if "cobrade" in c), None)
    if col_cobrade:
        COBRADE_HIDRO = ["12200", "12300", "12400", "13114", "13115"]
        df = df[df[col_cobrade].astype(str).str[:5].isin(COBRADE_HIDRO)].copy()
        print(f"[S2ID] {len(df)} eventos hidrológicos após filtro COBRADE.")

    col_mun = next((c for c in df.columns if "municipio" in c), "municipio")
    col_uf  = next((c for c in df.columns if c in ("uf", "sg_uf", "estado")), "uf")
    grp = ["cod_ibge"] + [c for c in [col_mun, col_uf] if c in df.columns]

    agg = {"total_ocorrencias": ("cod_ibge", "count")}
    if "afetados" in df.columns: agg["total_afetados"] = ("afetados", "sum")
    if "mortos"   in df.columns: agg["total_mortos"]   = ("mortos",   "sum")

    df_agg = df.groupby(grp).agg(**agg).reset_index()
    df_agg = df_agg.rename(
        columns={col_mun: "municipio", col_uf: "uf"}, errors="ignore"
    )
    df_agg["data_coleta"] = datetime.utcnow().isoformat()
    print(f"[S2ID] {len(df_agg)} municípios agregados.")
    return df_agg


# ── CadÚnico ──────────────────────────────────────────────────────────────────
def carregar_cadunico(caminho_zip: Path) -> pd.DataFrame:
    """
    Lê microdados amostrais desidentificados do CadÚnico (SAGICAD/MDS).
    Agrega por cod_ibge — nenhum dado individual é retido.
    """
    with zipfile.ZipFile(caminho_zip, "r") as z:
        print(f"[CadÚnico] Arquivos: {z.namelist()}")
        arq = next(
            (n for n in z.namelist()
             if "familia" in n.lower() and n.endswith((".csv", ".txt"))),
            next(n for n in z.namelist() if n.endswith((".csv", ".txt")))
        )
        print(f"[CadÚnico] Lendo: {arq}")
        with z.open(arq) as f:
            df = pd.read_csv(f, sep=";", encoding="latin-1", dtype=str)

    df.columns = [_norm(c) for c in df.columns]
    print(f"[CadÚnico] Primeiras colunas: {list(df.columns[:12])}")

    col_ibge = next(
        (c for c in df.columns
         if c in ("cd_ibge", "co_ibge", "cod_ibge", "co_municipio_ibge")),
        next((c for c in df.columns if "ibge" in c or "municipio" in c), None)
    )
    if col_ibge is None:
        raise ValueError(f"IBGE não encontrado. Colunas: {list(df.columns)}")

    df = df.rename(columns={col_ibge: "cod_ibge"})
    df["cod_ibge"] = df["cod_ibge"].str.strip().str.zfill(7)

    mapa_num = {
        "vlr_renda_media_fam":           "renda_media_pc",
        "cod_agua_canalizada_dom":        "cod_agua",
        "cod_escoa_sanitario_dom_comodo": "cod_esgoto",
        "qtd_pessoas_domic_fam":          "media_moradores",
    }
    for orig, novo in mapa_num.items():
        if orig in df.columns:
            df[novo] = pd.to_numeric(
                df[orig].str.replace(",", "."), errors="coerce"
            )

    agg = {"total_familias": ("cod_ibge", "count")}
    if "renda_media_pc"  in df.columns: agg["renda_media_pc"]  = ("renda_media_pc",  "mean")
    if "media_moradores" in df.columns: agg["media_moradores"] = ("media_moradores", "mean")
    if "cod_agua" in df.columns:
        agg["pct_sem_agua"] = ("cod_agua",
            lambda x: (pd.to_numeric(x, errors="coerce") != 1).mean() * 100)
    if "cod_esgoto" in df.columns:
        agg["pct_sem_esgoto"] = ("cod_esgoto",
            lambda x: pd.to_numeric(x, errors="coerce").isin([3, 4, 5]).mean() * 100)

    df_agg = df.groupby("cod_ibge").agg(**agg).reset_index()
    print(f"[CadÚnico] {len(df_agg)} municípios agregados.")
    return df_agg


# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":

    # ── IVS ──────────────────────────────────────────────────────────────────
    df_ivs = carregar_ivs(IVS_XLSX_PATH)
    df_ivs.to_csv(RAW_DIR / "ivs_tratado.csv", index=False)
    print(f"[OK] IVS salvo: {len(df_ivs)} municípios\n")

    # ── CadÚnico ─────────────────────────────────────────────────────────────
    cad_zip = download_arquivo(
        CADUNICO_URL, RAW_DIR / "cadunico_201812.zip", "CadÚnico/SAGI (dez/2018)"
    )
    df_cad = carregar_cadunico(cad_zip)
    df_cad.to_csv(RAW_DIR / "cadunico_agregado.csv", index=False)
    print(f"[OK] CadÚnico salvo: {len(df_cad)} municípios\n")

    # ── S2ID ─────────────────────────────────────────────────────────────────
    # s2id_path = RAW_DIR / "s2id_serie_historica.csv"
    # if s2id_path.exists():
    #     df_s2id = carregar_s2id(s2id_path)
    #     df_s2id.to_csv(RAW_DIR / "s2id_agregado.csv", index=False)
    #     print(f"[OK] S2ID salvo: {len(df_s2id)} municípios\n")
    # else:
    #     print(
    #         "[AVISO] S2ID não encontrado.\n"
    #         "  → Acesse https://s2id.mi.gov.br/paginas/series/\n"
    #         "  → Exporte como CSV e salve em data/raw/s2id_serie_historica.csv\n"
    #     )

    print("[CONCLUÍDO]")

[IVS] Lendo atlasivs_dadosbrutos_pt_v2.xlsx (pode demorar alguns segundos)...
[IVS] Shape bruto: (340786, 103)
[IVS] Após filtro nivel=municipio: 297437 linhas
[IVS] Ano selecionado: 2010 | 149576 linhas
[IVS] Após filtro Total Cor: 49878 linhas
[IVS] 5565 municípios únicos | colunas: ['cod_ibge', 'ivs_geral', 'ivs_infraestrutura_urbana', 'ivs_capital_humano', 'ivs_renda_trabalho']
[IVS] Amostra:
      cod_ibge  ivs_geral  ivs_infraestrutura_urbana  ivs_capital_humano  ivs_renda_trabalho
40666  5300108      0.294                      0.412               0.265               0.204
40668  5200134      0.314                      0.245               0.369               0.328
40670  5200159      0.311                      0.204               0.287               0.441

[OK] IVS salvo: 5565 municípios

[DOWNLOAD] CadÚnico/SAGI (dez/2018) ...
[OK] 378602 KB
[CadÚnico] Arquivos: ['base_amostra_cad_201812/', 'base_amostra_cad_201812/base_amostra_familia_201812.csv', 'base_amostra_cad_201812/base_

In [3]:
"""
============================================================
ETAPA 3 — Governança, Mascaramento e Conformidade LGPD
Pipeline: Priorização de Infraestrutura Urbana Preventiva

Técnicas aplicadas:
  - Pseudoanonimização com SHA-256 + salt rotativo
  - Agregação territorial por município (cod_ibge)
  - Remoção de quasi-identificadores
  - Supressão de células com contagem < k (k-anonimato)

Entrada:  data/raw/cadunico_201812.zip   (baixado pelo 01_ingestao.py)
Saída:    data/processed/cadunico_governado.csv

Requer: pip install pandas
============================================================
"""

import hashlib
import zipfile
import unicodedata
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

# ── Diretórios ────────────────────────────────────────────────────────────────
RAW_DIR       = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# ── Configurações ─────────────────────────────────────────────────────────────
# Salt rotativo mensal — impede correlação entre execuções de períodos distintos
SALT        = f"enap2025_{datetime.now(timezone.utc).strftime('%Y%m')}"
K_ANONIMATO = 5   # mínimo de famílias por célula analítica

# Colunas com dados pessoais a remover antes de qualquer análise
COLUNAS_REMOVER = [
    "nom_pessoa", "nom_mae", "nom_pai", "dat_nasc",
    "num_cpf_pessoa", "num_nis_pessoa", "num_telefone",
    "nom_tip_logradouro", "nom_logradouro", "num_cep",
    "num_comodo_servicos", "cod_familia",
]


# ── Utilitário ────────────────────────────────────────────────────────────────
def _norm(s: str) -> str:
    s = str(s).strip().lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return s.replace(" ", "_").replace("-", "_").replace("/", "_")


# ── 1. Pseudoanonimização ─────────────────────────────────────────────────────
def pseudoanonimizar(valor: str) -> str:
    """Hash SHA-256 com salt. Reproduzível no mês, não rastreável entre períodos."""
    token = f"{SALT}:{str(valor)}".encode("utf-8")
    return hashlib.sha256(token).hexdigest()[:12]


# ── 2. Leitura do ZIP do CadÚnico ────────────────────────────────────────────
def ler_cadunico_zip(caminho_zip: Path) -> pd.DataFrame:
    """
    Lê o arquivo de famílias do ZIP do CadÚnico (SAGICAD/MDS).
    Mantém apenas colunas necessárias para a análise territorial.
    """
    with zipfile.ZipFile(caminho_zip, "r") as z:
        print(f"[CadÚnico] Arquivos no ZIP: {z.namelist()}")
        arq = next(
            (n for n in z.namelist()
             if "familia" in n.lower() and n.endswith((".csv", ".txt"))),
            next(n for n in z.namelist() if n.endswith((".csv", ".txt")))
        )
        print(f"[CadÚnico] Lendo: {arq}")
        with z.open(arq) as f:
            df = pd.read_csv(f, sep=";", encoding="latin-1", dtype=str)

    df.columns = [_norm(c) for c in df.columns]
    print(f"[CadÚnico] Shape bruto: {df.shape}")
    print(f"[CadÚnico] Primeiras colunas: {list(df.columns[:15])}")
    return df


# ── 3. Remoção de identificadores ────────────────────────────────────────────
def remover_identificadores(df: pd.DataFrame) -> pd.DataFrame:
    """Remove colunas com dados pessoais diretos e quasi-identificadores."""
    existentes = [c for c in COLUNAS_REMOVER if c in df.columns]
    df = df.drop(columns=existentes)
    print(f"[LGPD] {len(existentes)} colunas identificadoras removidas: {existentes}")
    return df


# ── 4. Agregação por município com k-anonimato ───────────────────────────────
def agregar_por_municipio(df: pd.DataFrame) -> pd.DataFrame:
    """
    Detecta a coluna de código IBGE, converte colunas numéricas
    e agrega os microdados em estatísticas por município.
    Aplica k-anonimato: suprime municípios com menos de K_ANONIMATO famílias.
    """
    # Detecta coluna IBGE
    col_ibge = next(
        (c for c in df.columns
         if c in ("cd_ibge", "co_ibge", "cod_ibge", "co_municipio_ibge")),
        next((c for c in df.columns if "ibge" in c or "municipio" in c), None)
    )
    if col_ibge is None:
        raise ValueError(f"Coluna IBGE não encontrada. Colunas: {list(df.columns)}")

    df = df.rename(columns={col_ibge: "cod_ibge"})
    df["cod_ibge"] = df["cod_ibge"].astype(str).str.strip().str.zfill(7)

    # Converte colunas numéricas relevantes
    mapa_num = {
        "vlr_renda_media_fam":           "renda_media_pc",
        "qtd_pessoas_domic_fam":          "media_moradores",
        "cod_agua_canalizada_dom":        "cod_agua",
        "cod_escoa_sanitario_dom_comodo": "cod_esgoto",
    }
    for orig, novo in mapa_num.items():
        if orig in df.columns:
            df[novo] = pd.to_numeric(
                df[orig].str.replace(",", "."), errors="coerce"
            )

    # Monta agregação dinamicamente conforme colunas disponíveis
    agg = {"total_familias": ("cod_ibge", "count")}
    if "renda_media_pc"  in df.columns:
        agg["renda_media_pc"]  = ("renda_media_pc",  "mean")
    if "media_moradores" in df.columns:
        agg["media_moradores"] = ("media_moradores", "mean")
    if "cod_agua" in df.columns:
        agg["pct_sem_agua"] = (
            "cod_agua",
            lambda x: round((pd.to_numeric(x, errors="coerce") != 1).mean() * 100, 2)
        )
    if "cod_esgoto" in df.columns:
        agg["pct_sem_esgoto"] = (
            "cod_esgoto",
            lambda x: round(
                pd.to_numeric(x, errors="coerce").isin([3, 4, 5]).mean() * 100, 2
            )
        )

    df_agg = df.groupby("cod_ibge").agg(**agg).reset_index()

    # k-anonimato
    antes = len(df_agg)
    df_agg = df_agg[df_agg["total_familias"] >= K_ANONIMATO].copy()
    suprimidos = antes - len(df_agg)
    print(f"[k-anon] {suprimidos} município(s) suprimido(s) (< {K_ANONIMATO} famílias).")
    print(f"[Agregação] {len(df_agg)} municípios | colunas: {list(df_agg.columns)}")
    return df_agg


# ── 5. Validação de conformidade ─────────────────────────────────────────────
def validar_conformidade(df: pd.DataFrame) -> dict:
    """Verifica se o DataFrame está em conformidade com a LGPD."""
    quasi_id_restantes  = [c for c in df.columns if c in COLUNAS_REMOVER]
    campos_pessoais     = [
        c for c in df.columns
        if any(p in c for p in ["cpf", "nis", "nom_", "nasc", "telefone", "cep"])
    ]
    k_min = int(df["total_familias"].fillna(0).min()) \
            if "total_familias" in df.columns else K_ANONIMATO

    relatorio = {
        "timestamp":                datetime.now(timezone.utc).isoformat(),
        "total_registros":          len(df),
        "quasi_id_restantes":       quasi_id_restantes,
        "campos_pessoais_detectados": campos_pessoais,
        "k_anonimato_minimo":       k_min,
        "conforme": len(quasi_id_restantes) == 0 and len(campos_pessoais) == 0,
    }

    status = "CONFORME" if relatorio["conforme"] else "REQUER REVISAO"
    print(f"\n[LGPD] Relatorio de Conformidade: {status}")
    for k, v in relatorio.items():
        print(f"  {k}: {v}")
    return relatorio


# ── Main ──────────────────────────────────────────────────────────────────────
if __name__ == "__main__":

    # Localiza o ZIP do CadÚnico (aceita qualquer versão baixada)
    zips = sorted(RAW_DIR.glob("cadunico_*.zip"))
    if not zips:
        raise FileNotFoundError(
            "Nenhum arquivo cadunico_*.zip encontrado em data/raw/\n"
            "Execute primeiro o 01_ingestao.py para baixar o CadUnico."
        )
    caminho_zip = zips[-1]   # usa o mais recente
    print(f"[CadÚnico] Usando: {caminho_zip.name}\n")

    # Pipeline completo
    df_bruto    = ler_cadunico_zip(caminho_zip)
    df_sem_id   = remover_identificadores(df_bruto)
    df_agregado = agregar_por_municipio(df_sem_id)
    relatorio   = validar_conformidade(df_agregado)

    if not relatorio["conforme"]:
        raise RuntimeError(
            f"[LGPD] Pipeline interrompido — dados nao conformes.\n"
            f"Campos detectados: {relatorio['campos_pessoais_detectados']}"
        )

    # Salva resultado
    saida = PROCESSED_DIR / "cadunico_governado.csv"
    df_agregado.to_csv(saida, index=False)
    print(f"\n[OK] {len(df_agregado)} municipios salvos em {saida}")

[CadÚnico] Usando: cadunico_201812.zip

[CadÚnico] Arquivos no ZIP: ['base_amostra_cad_201812/', 'base_amostra_cad_201812/base_amostra_familia_201812.csv', 'base_amostra_cad_201812/base_amostra_pessoa_201812.csv']
[CadÚnico] Lendo: base_amostra_cad_201812/base_amostra_familia_201812.csv
[CadÚnico] Shape bruto: (4807996, 31)
[CadÚnico] Primeiras colunas: ['cd_ibge', 'estrato', 'classf', 'id_familia', 'dat_cadastramento_fam', 'dat_alteracao_fam', 'vlr_renda_media_fam', 'dat_atualizacao_familia', 'cod_local_domic_fam', 'cod_especie_domic_fam', 'qtd_comodos_domic_fam', 'qtd_comodos_dormitorio_fam', 'cod_material_piso_fam', 'cod_material_domic_fam', 'cod_agua_canalizada_fam']
[LGPD] 0 colunas identificadoras removidas: []
[k-anon] 1 município(s) suprimido(s) (< 5 famílias).
[Agregação] 5533 municípios | colunas: ['cod_ibge', 'total_familias', 'renda_media_pc']

[LGPD] Relatorio de Conformidade: CONFORME
  timestamp: 2026-03-29T13:41:53.166272+00:00
  total_registros: 5533
  quasi_id_restant

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

Path("data/processed").mkdir(parents=True, exist_ok=True)

# ── Carrega as 2 bases disponíveis ───────────────────────────────────────────
df_cad = pd.read_csv("data/processed/cadunico_governado.csv", dtype={"cod_ibge": str})
df_ivs = pd.read_csv("data/raw/ivs_tratado.csv",              dtype={"cod_ibge": str})

print(f"CadÚnico: {len(df_cad)} municípios | colunas: {list(df_cad.columns)}")
print(f"IVS:      {len(df_ivs)} municípios | colunas: {list(df_ivs.columns)}")

# ── Cruzamento ────────────────────────────────────────────────────────────────
df = df_cad.merge(df_ivs, on="cod_ibge", how="inner")
print(f"\nApós cruzamento: {len(df)} municípios")

# ── Converte numéricos ────────────────────────────────────────────────────────
num_cols = ["total_familias","renda_media_pc",
            "ivs_geral","ivs_infraestrutura_urbana",
            "ivs_capital_humano","ivs_renda_trabalho",
            "pct_sem_agua","pct_sem_esgoto"]
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ── Score (sem S2ID por enquanto — peso redistribuído) ────────────────────────
# IVS infraestrutura urbana: 55% | Vulnerabilidade sanitária CadÚnico: 45%
def norm(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

col_ivs = "ivs_infraestrutura_urbana" if "ivs_infraestrutura_urbana" in df.columns else "ivs_geral"
score_ivs = norm(df[col_ivs].fillna(df[col_ivs].median()))

if "pct_sem_agua" in df.columns and "pct_sem_esgoto" in df.columns:
    score_san = 0.5*norm(df["pct_sem_agua"].fillna(0)) + 0.5*norm(df["pct_sem_esgoto"].fillna(0))
else:
    # Usa renda inversa como proxy de vulnerabilidade sanitária
    score_san = norm(1 / (df["renda_media_pc"].fillna(df["renda_media_pc"].median()) + 1))

df["score_prioridade"] = (0.55*score_ivs + 0.45*score_san).mul(100).round(2)

df["faixa_prioridade"] = pd.cut(
    df["score_prioridade"],
    bins=[0, 25, 50, 75, 100],
    labels=["BAIXA","MEDIA","ALTA","CRITICA"],
    include_lowest=True
).astype(str)

# ── Resultados ────────────────────────────────────────────────────────────────
print("\n── Distribuição por faixa ──")
print(df["faixa_prioridade"].value_counts().sort_index())

print("\n── Top 10 prioritários ──")
cols_disp = [c for c in ["cod_ibge","renda_media_pc",col_ivs,
             "pct_sem_agua","pct_sem_esgoto","score_prioridade","faixa_prioridade"]
             if c in df.columns]
print(df.sort_values("score_prioridade", ascending=False)[cols_disp].head(10).to_string(index=False))

# ── Salva ─────────────────────────────────────────────────────────────────────


df_sorted = df.sort_values("score_prioridade", ascending=False).reset_index(drop=True)
df_sorted.to_csv("data/processed/municipios_priorizados.csv", index=False)

zonas = df_sorted[df_sorted["faixa_prioridade"].isin(["CRITICA","ALTA"])]
zonas.to_csv("data/processed/zonas_vermelhas.csv", index=False)

# ── PERSISTE NO PATH DO DRIVE  ─────────────────────────────────────────────────────────────────────
DRIVE_PROCESSED = Path('/content/drive/MyDrive/ENAP/dados_pnpdec')
DRIVE_PROCESSED.mkdir(parents=True, exist_ok=True)

df_sorted.to_csv(DRIVE_PROCESSED / 'municipios_priorizados.csv', index=False)
zonas.to_csv(DRIVE_PROCESSED / 'zonas_vermelhas.csv', index=False)


print(f"\n[OK] {len(df_sorted)} municípios priorizados salvos")
print(f"[OK] {len(zonas)} zonas CRÍTICA/ALTA exportadas")
print("\nNota: score provisório sem S2ID (55% IVS + 45% Saneamento).")
print("Quando o S2ID estiver disponível, reprocessar com os 3 componentes.")

CadÚnico: 5533 municípios | colunas: ['cod_ibge', 'total_familias', 'renda_media_pc']
IVS:      5565 municípios | colunas: ['cod_ibge', 'ivs_geral', 'ivs_infraestrutura_urbana', 'ivs_capital_humano', 'ivs_renda_trabalho']

Após cruzamento: 5529 municípios

── Distribuição por faixa ──
faixa_prioridade
ALTA       30
BAIXA    4823
MEDIA     676
Name: count, dtype: int64

── Top 10 prioritários ──
cod_ibge  renda_media_pc  ivs_infraestrutura_urbana  score_prioridade faixa_prioridade
 1506005       91.713287                      0.992             59.70             ALTA
 2103125      102.889610                      1.000             59.53             ALTA
 2110278       78.354651                      0.964             59.11             ALTA
 2105104      116.397914                      1.000             58.95             ALTA
 2105005      126.442451                      1.000             58.60             ALTA
 2112407       80.102743                      0.948             58.09           

In [5]:
import pandas as pd
import requests

# ── Busca nomes via API IBGE ──────────────────────────────────────────────────
print("Buscando municipios no IBGE...")
resp = requests.get(
    "https://servicodados.ibge.gov.br/api/v1/localidades/municipios?view=nivelado",
    timeout=30
)
df_ibge = pd.DataFrame(resp.json())
df_ibge["cod_ibge"] = df_ibge["municipio-id"].astype(str)
df_ibge = df_ibge[["cod_ibge","municipio-nome","UF-sigla"]].rename(
    columns={"municipio-nome":"nome_municipio","UF-sigla":"uf"}
)
print(f"IBGE: {len(df_ibge)} municipios carregados")

# ── Enriquece ─────────────────────────────────────────────────────────────────
df = pd.read_csv("data/processed/municipios_priorizados.csv", dtype={"cod_ibge": str})
df = df.merge(df_ibge, on="cod_ibge", how="left")
df.to_csv("data/processed/municipios_priorizados.csv", index=False)

zonas = df[df["faixa_prioridade"].isin(["CRITICA","ALTA"])].copy()
zonas.to_csv("data/processed/zonas_vermelhas.csv", index=False)

# ── Top 15 ───────────────────────────────────────────────────────────────────
print("\nTop 15 zonas prioritarias (dados reais):")
cols = ["nome_municipio","uf","renda_media_pc",
        "ivs_infraestrutura_urbana","score_prioridade","faixa_prioridade"]
cols_ok = [c for c in cols if c in zonas.columns]
print(zonas[cols_ok].head(15).to_string(index=False))

print("\nDistribuicao por UF (zonas ALTA/CRITICA):")
print(zonas["uf"].value_counts().head(10).to_string())

print("\nEstatisticas gerais:")
print(f"  Municipios analisados : {len(df)}")
print(f"  Zonas CRITICA/ALTA    : {len(zonas)}")
print(f"  IVS medio total       : {df['ivs_infraestrutura_urbana'].mean():.3f}")
print(f"  IVS medio zonas ALTA  : {zonas['ivs_infraestrutura_urbana'].mean():.3f}")
print(f"  Renda media PC total  : R$ {df['renda_media_pc'].mean():.2f}")
print(f"  Renda media PC ALTA   : R$ {zonas['renda_media_pc'].mean():.2f}")

Buscando municipios no IBGE...
IBGE: 5571 municipios carregados

Top 15 zonas prioritarias (dados reais):
         nome_municipio uf  renda_media_pc  ivs_infraestrutura_urbana  score_prioridade faixa_prioridade
                Prainha PA       91.713287                      0.992             59.70             ALTA
    Central do Maranhão MA      102.889610                      1.000             59.53             ALTA
Santo Amaro do Maranhão MA       78.354651                      0.964             59.11             ALTA
                  Icatu MA      116.397914                      1.000             58.95             ALTA
     Humberto de Campos MA      126.442451                      1.000             58.60             ALTA
                Turiaçu MA       80.102743                      0.948             58.09             ALTA
      Amapá do Maranhão MA      133.584906                      0.987             57.67             ALTA
                 Bacuri MA      193.281955            